# Part 2: License Plate Enhancement, Sharpening & Character Recognition (OCR)

**Project:** Vehicle Number Plate Image Enhancement and Sharpening System Using Computer Vision  
**Components Covered in this Notebook:**
1. **Blur Detection & Quality Analysis** (Laplacian Variance, Tenengrad, FFT)
2. **Controlled Degradation Simulation** (Motion blur, Gaussian defocus, sensor noise)
3. **Noise Reduction Filtering** (Gaussian vs Median vs Bilateral filter)
4. **Sharpening Algorithms Comparison** (Laplacian vs High-Pass Filter vs Unsharp Masking)
5. **Image Quality Metrics** (PSNR, SSIM, CNR, Entropy)
6. **Optical Character Recognition (OCR)** (EasyOCR extraction, HSRP 'IND' emblem handling, Indian plate format validation)
7. **OCR Accuracy Evaluation** (Character Error Rate - CER, Levenshtein edit distance, Exact Match)
8. **End-to-End Pipeline Demonstration**

In [ ]:
import sys
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.enhancement.blur_detector import detect_blur
from src.enhancement.degradation import simulate_degradation_pipeline, apply_motion_blur, apply_gaussian_noise
from src.enhancement.noise_reduction import apply_gaussian_filter, apply_median_filter, apply_bilateral_filter
from src.enhancement.sharpening import laplacian_sharpen, highpass_sharpen, unsharp_mask, compare_sharpening_methods
from src.evaluation.metrics import evaluate_image_quality, compute_ocr_accuracy
from src.ocr.reader import recognize_plate_text
from src.pipeline import PlateEnhancementPipeline

print("All modules imported successfully!")

## 1. Load a Sample License Plate from Dataset
We load a real cropped license plate (`UP84AE9889`) extracted from the dataset.

In [ ]:
sample_path = PROJECT_ROOT / "data" / "sample_plates" / "plate_dc_auto_image_000024_fqvRhfiO6i_0_UP84AE9889.jpg"
ground_truth = "UP84AE9889"

plate_img = cv2.imread(str(sample_path))
assert plate_img is not None, f"Sample image not found: {sample_path}"

plt.figure(figsize=(6, 3))
plt.imshow(cv2.cvtColor(plate_img, cv2.COLOR_BGR2RGB))
plt.title(f"Ground Truth Plate: {ground_truth} (Dimensions: {plate_img.shape[1]}x{plate_img.shape[0]})")
plt.axis("off")
plt.show()

## 2. Blur Detection & Sharpness Analysis
We measure sharpness using **Laplacian Variance** $\text{Var}(\nabla^2 I)$, **Tenengrad Energy**, and **FFT Frequency Ratio**.

In [ ]:
blur_results = detect_blur(plate_img)
print("Blur Detection Results:")
for k, v in blur_results.items():
    print(f"  {k:20}: {v}")

## 3. Controlled Degradation Simulation
To scientifically benchmark our enhancement pipeline, we simulate realistic CCTV and speed camera degradation:
- Motion Blur (simulating vehicle movement)
- Gaussian Electronic Noise

In [ ]:
degraded_img = simulate_degradation_pipeline(plate_img, degradation_type="motion_and_noise", severity="medium")
deg_blur = detect_blur(degraded_img)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.cvtColor(plate_img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original Plate\nLaplacian Var: {blur_results['laplacian_var']}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(degraded_img, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Degraded Plate (Motion + Noise)\nLaplacian Var: {deg_blur['laplacian_var']}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Noise Reduction Filtering (Gaussian vs Median vs Bilateral)
Notice how the **Bilateral Filter** preserves character edge transitions while smoothing flat plate metal noise.

In [ ]:
gaussian = apply_gaussian_filter(degraded_img, kernel_size=3, sigma=1.0)
median = apply_median_filter(degraded_img, kernel_size=3)
bilateral = apply_bilateral_filter(degraded_img, d=7, sigma_color=50, sigma_space=50)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(degraded_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Degraded Input")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(gaussian, cv2.COLOR_BGR2RGB))
axes[1].set_title("Gaussian Filter (3x3)")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(median, cv2.COLOR_BGR2RGB))
axes[2].set_title("Median Filter (3x3)")
axes[2].axis("off")

axes[3].imshow(cv2.cvtColor(bilateral, cv2.COLOR_BGR2RGB))
axes[3].set_title("Bilateral Filter (Edge-Preserving)")
axes[3].axis("off")
plt.tight_layout()
plt.show()

## 5. Sharpening Comparison: Laplacian vs High-Pass vs Unsharp Masking
We apply sharpening on the Bilateral-filtered image:

In [ ]:
lap_img = laplacian_sharpen(bilateral, alpha=0.6)
hpf_img = highpass_sharpen(bilateral, beta=0.8)
usm_img = unsharp_mask(bilateral, amount=1.6, threshold=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cv2.cvtColor(lap_img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Laplacian Sharpening (\u03b1=0.6)\nLap. Var: {detect_blur(lap_img)['laplacian_var']:.1f}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(hpf_img, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"High-Pass Sharpening (\u03b2=0.8)\nLap. Var: {detect_blur(hpf_img)['laplacian_var']:.1f}")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(usm_img, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Unsharp Masking (Amount=1.6)\nLap. Var: {detect_blur(usm_img)['laplacian_var']:.1f}")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 6. OCR Text Extraction & Metric Evaluation
We compare OCR character recognition on the Degraded Plate vs the Enhanced Plate:

In [ ]:
raw_ocr = recognize_plate_text(degraded_img)
enh_ocr = recognize_plate_text(usm_img)

raw_acc = compute_ocr_accuracy(ground_truth, raw_ocr['text'])
enh_acc = compute_ocr_accuracy(ground_truth, enh_ocr['text'])

print(f"Ground Truth Plate Number : {ground_truth}")
print("-" * 50)
print(f"Degraded Plate OCR       : '{raw_ocr['text']}' | Confidence: {raw_ocr['confidence']:.2f} | CER: {raw_acc['cer']:.3f}")
print(f"Enhanced Plate OCR       : '{enh_ocr['text']}' | Confidence: {enh_ocr['confidence']:.2f} | CER: {enh_acc['cer']:.3f}")
print(f"Exact Match              : {'YES \u2705' if enh_acc['exact_match'] else 'NO'}")

quality_metrics = evaluate_image_quality(usm_img, reference=plate_img)
print("-" * 50)
print(f"Restoration PSNR (dB)    : {quality_metrics.get('psnr_db', 0):.2f} dB")
print(f"Restoration SSIM         : {quality_metrics.get('ssim', 0):.4f}")
print(f"Contrast-to-Noise (CNR)  : {quality_metrics.get('cnr', 0):.2f}")

## 7. Full End-to-End Pipeline Execution
Running our complete `PlateEnhancementPipeline` which automatically processes the image and creates a comprehensive report:

In [ ]:
pipeline = PlateEnhancementPipeline(sharpening_method="unsharp_mask", usm_amount=1.6)
result = pipeline.process(degraded_img, ground_truth=ground_truth)
fig_path = pipeline.generate_visualization(result, output_path=PROJECT_ROOT / "outputs" / "notebook_pipeline_demo.png")
print(f"Generated end-to-end figure: {fig_path}")

# Display saved figure
fig_img = cv2.imread(str(fig_path))
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(fig_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()